In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_DIR / "src"

sys.path.append(str(SRC_DIR))
print(PROJECT_DIR)

In [ ]:
from core.config import update_path_settings

update_path_settings(PROJECT_DIR)

In [ ]:
from core.setup import settings, setup_data

DOWNLOAD_DATA = False
if DOWNLOAD_DATA:
    setup_data()

In [ ]:
SPECIES_DIRS = [d for d in settings.DATA_RAW_DIR.iterdir() if d.is_dir()]
SPECIES_DIRS.sort()

LW_SPECIES_DIR = [d for d in SPECIES_DIRS if "LW" in d.name][0]

In [ ]:
from domain.pipelines.annotations import (
    append_metrics,
    df_to_annotations,
    filter_noise,
    load_annotation_file,
    normalize_header,
    normalize_species_and_calls,
    parse_numerics,
)
from domain.pipelines.audio import load_audio_torchaudio
from domain.pipelines.types import Annotation, AudioRecord

ANNOTATIONS_EXT = ".txt"
AUDIO_EXT = ".wav"
SAMPLE_RATE = 28000

recordings: list[AudioRecord] = []
for file in LW_SPECIES_DIR.iterdir():
    if file.suffix.lower() != AUDIO_EXT:
        continue

    audio_path = file
    annotation_df_path = file.with_suffix(ANNOTATIONS_EXT)
    annotations: list[Annotation] = []

    if annotation_df_path.exists():
        df = load_annotation_file(annotation_df_path)
        df = normalize_header(df)
        df = normalize_species_and_calls(df)
        df = parse_numerics(df)
        df = filter_noise(df)
        df = append_metrics(df)
        annotations = df_to_annotations(df)

    wav_tensor = load_audio_torchaudio(audio_path, sample_rate=SAMPLE_RATE)
    record = AudioRecord(
        wav=wav_tensor,
        sample_rate=SAMPLE_RATE,
        annotations=annotations,
    )
    recordings.append(record)

In [ ]:
import matplotlib.pyplot as plt

from domain.pipelines.annotations import annotations_to_df
from domain.pipelines.image import compute_spectrogram

NFFT = 1024
HOP = 256
sample_record = recordings[10]
spec = compute_spectrogram(sample_record.wav, n_fft=NFFT, hop_length=HOP)
plt.figure(figsize=(20, 4))
plt.imshow(spec[0].numpy(), aspect="auto", origin="lower")
plt.title(f"Spectrogram {sample_record.sample_rate}Hz")
plt.xlabel("Time")
plt.ylabel("Frequency")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
for ann in sample_record.annotations:
    x1 = int(ann.begin_time * sample_record.sample_rate / HOP)
    x2 = int(ann.end_time * sample_record.sample_rate / HOP)
    y1 = int(ann.low_freq * NFFT / sample_record.sample_rate)
    y2 = int(ann.high_freq * NFFT / sample_record.sample_rate)
    plt.gca().add_patch(
        plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            edgecolor="red",
            facecolor="none",
            linewidth=2,
        )
    )

plt.show()

df = annotations_to_df(sample_record.annotations)
df

In [ ]:
from domain.pipelines.pipeline import SpectrogramDataset, WindowConfig

window_cfg = WindowConfig(
    duration_sec=3.0,
    hop_sec=1.5,
    n_fft=NFFT,
    hop_length=HOP,
    sample_rate=SAMPLE_RATE,
    img_size=640,
    channel_methods=("min_max", "z_score_per_band", "noise_filtered"),
)


def class_mapping_fn(species: str, call_type: str) -> int:
    mapping = {
        "cs": 0,  # call_syllable
        "cc": 1,  # call_cluster/phrase
    }
    return mapping.get(call_type, 2)

In [ ]:
from sklearn.model_selection import train_test_split

from domain.pipelines.pipeline import export_to_yolo, write_data_yaml

train_recs, val_recs = train_test_split(
    recordings,
    test_size=0.2,
    random_state=42,
)

ds_train = SpectrogramDataset(
    recordings=train_recs, cfg=window_cfg, class_mapping_fn=class_mapping_fn
)

ds_val = SpectrogramDataset(recordings=val_recs, cfg=window_cfg, class_mapping_fn=class_mapping_fn)

YOLO_PATH = settings.DATA_DIR / "yolo"

export_to_yolo(ds_train, YOLO_PATH, split="train")
export_to_yolo(ds_val, YOLO_PATH, split="val")

In [ ]:
write_data_yaml(
    output_path=YOLO_PATH,
    train_split="train",
    val_split="val",
    names={
        0: "call_syllable",
        1: "call_cluster",
        2: "other",
    },
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    cache=False,
    name="yolo_primate",
    data=YOLO_PATH / "data.yaml",
    imgsz=640,
    multi_scale=True,
    overlap_mask=False,
    batch=8,
    workers=4,
)

In [ ]:
from domain.pipelines.inference import BioacousticInference

test_path: Path = Path(settings.DATA_DIR) / "test" / "240125_0028.wav"
detections_path: Path = test_path.parent / f"{test_path.stem}_detections.txt"
model_path = Path(settings.PROJECT_DIR) / "runs" / "detect" / "yolo_primate" / "weights" / "best.pt"

inference = BioacousticInference(model_path, window_cfg)

df_results = inference.run(test_path)
df_results.to_csv(detections_path, index=False, sep="\t")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torchaudio

from domain.pipelines.image import compute_spectrogram

wav, orig_freq = torchaudio.load(test_path, normalize=True)
if orig_freq != SAMPLE_RATE:
    resampled_wav = torchaudio.transforms.Resample(orig_freq=orig_freq, new_freq=SAMPLE_RATE)(
        wav[0]
    )
else:
    resampled_wav = wav[0]

spec = compute_spectrogram(resampled_wav, n_fft=NFFT, hop_length=HOP)
df_results = pd.read_csv(detections_path, sep="\t")
plt.figure(figsize=(25, 6))
plt.imshow(spec[0].numpy(), aspect="auto", origin="lower")
plt.title(f"Inference Spectrogram {SAMPLE_RATE}Hz")
plt.xlabel("Time Frames")
plt.ylabel("Frequency Bins")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()

# Overlay each detection
for _, row in df_results.iterrows():
    x1 = int(row["begin_time"] * SAMPLE_RATE / HOP)
    x2 = int(row["end_time"] * SAMPLE_RATE / HOP)
    y1 = int(row["low_freq"] * NFFT / SAMPLE_RATE)
    y2 = int(row["high_freq"] * NFFT / SAMPLE_RATE)

    plt.gca().add_patch(
        plt.Rectangle(
            (x1, y1), x2 - x1, y2 - y1, edgecolor="lime", facecolor="none", linewidth=2, alpha=0.8
        )
    )
    plt.text(
        x1,
        y2 + 2,
        f"{row['species']} ({row.get('Rating', '')})" if "Rating" in row else row["species"],
        color="lime",
        fontsize=10,
        backgroundcolor="black",
    )

plt.show()